In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ami0nai/quran-dataset/quran_full_data_collection_19_1008_murattal/quran_metadata.json
/kaggle/input/datasets/ami0nai/quran-dataset/quran_full_data_collection_19_1008_murattal/audio/Ayman_Sowaid_64kbps_s102_a5.mp3
/kaggle/input/datasets/ami0nai/quran-dataset/quran_full_data_collection_19_1008_murattal/audio/Ali_Jaber_64kbps_s104_a4.mp3
/kaggle/input/datasets/ami0nai/quran-dataset/quran_full_data_collection_19_1008_murattal/audio/Muhsin_Al_Qasim_192kbps_s102_a6.mp3
/kaggle/input/datasets/ami0nai/quran-dataset/quran_full_data_collection_19_1008_murattal/audio/Ayman_Sowaid_64kbps_s98_a7.mp3
/kaggle/input/datasets/ami0nai/quran-dataset/quran_full_data_collection_19_1008_murattal/audio/Hani_Rifai_192kbps_s98_a5.mp3
/kaggle/input/datasets/ami0nai/quran-dataset/quran_full_data_collection_19_1008_murattal/audio/Muhsin_Al_Qasim_192kbps_s94_a8.mp3
/kaggle/input/datasets/ami0nai/quran-dataset/quran_full_data_collection_19_1008_murattal/audio/Hani_Rifai_192kbps_s97_a2.mp3
/ka

In [5]:
# ==============================================================================
# CELL 1: System Dependencies & Python Libraries
# ==============================================================================
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1 portaudio19-dev

# Install pinned compatible packages
!pip install "numpy<2.3" --quiet
!pip install torch torchaudio transformers librosa soundfile python-Levenshtein \
             fuzzysearch pydantic xmltodict rapidfuzz tqdm matplotlib pandas seaborn --quiet
!pip install quran-transcript==0.5.2 --no-deps --quiet
!pip install scikit-learn --quiet

print("✅ Environment setup complete.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
dpkg: libjack-jackd2-0:amd64: dependency problems, but removing anyway as you requested:
 libavdevice58:amd64 depends on libjack-jackd2-0 (>= 1.9.10+20150825) | libjack-0.125; however:
  Package libjack-jackd2-0:amd64 is to be removed.
  Package libjack-0.125 is not installed.
  Package libjack-jackd2-0:amd64 which provides libjack-0.125 is to be removed.
 libavdevice58:amd64 depends on libjack-jackd2-0 (>= 1.9.10+20150825) | libjack-0.125; however:
  Package libjack-jackd2-0:amd64 is to be removed.
  Package libjack-0.125 is not installed.
  Package libjack-jackd2-0:amd64 which provides libjack-0.125 is to be removed.

(Reading database ... 125186 files and directories currently installed.)
Removing libjack-jackd2-0:amd64 (1.9.20~dfsg-1) ...
Selecting previously unselected package libjack0:amd64.
(R

In [6]:
# ==============================================================================
# CELL 2: Imports & Quran-Transcript Patching
# ==============================================================================
import os
import re
import sys
import json
import importlib
import subprocess
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoFeatureExtractor, AutoModel
import quran_transcript
from tqdm.auto import tqdm

# --- DEVICE SETUP ---
NUM_GPUS = torch.cuda.device_count()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using Compute Device: {DEVICE}")

# --- PATCH QURAN TRANSCRIPT ---
def patch_quran_transcript_sifa_on_disk():
    package_dir = os.path.dirname(quran_transcript.__file__)
    sifa_path = None
    for root, _, files in os.walk(package_dir):
        if "sifa.py" in files:
            sifa_path = os.path.join(root, "sifa.py")
            break

    if sifa_path is None: return

    with open(sifa_path, 'r', encoding='utf-8') as f: content = f.read()
    modified = False

    if "alif can not start a phoneme script" in content or "alif cannot start" in content:
        content = re.sub(r'raise\s+ValueError\s*\(\s*["\']alif can\s*not start a phoneme script["\']\s*\)', 'pass', content)
        modified = True

    if "alif_tafkheem_and_tarqeeq[alif_idx]" in content and "len(alif_tafkheem_and_tarqeeq)" not in content:
        content = content.replace("alif_tafkheem_and_tarqeeq[alif_idx]", "(alif_tafkheem_and_tarqeeq[alif_idx] if alif_idx < len(alif_tafkheem_and_tarqeeq) else 'tarqeeq')")
        modified = True

    if modified:
        with open(sifa_path, 'w', encoding='utf-8') as f: f.write(content)
        for mod in [m for m in sys.modules if m.startswith("quran_transcript")]:
            try: importlib.reload(sys.modules[mod])
            except Exception: pass
        print("🎉 Successfully patched quran-transcript on disk.")

patch_quran_transcript_sifa_on_disk()


🚀 Using Compute Device: cuda
🎉 Successfully patched quran-transcript on disk.


In [8]:
# ==============================================================================
# CELL 3: Deterministic Vocabulary & Normalizer
# ==============================================================================
class DeterministicVocabulary:
    def __init__(self, phoneme_list=None):
        self.phoneme2id = {"<pad>": 0, "<blank>": 1}
        self.id2phoneme = {0: "<pad>", 1: "<blank>"}

        if phoneme_list:
            sorted_tokens = sorted(list(set(phoneme_list)))
            for idx, p in enumerate(sorted_tokens, start=2):
                self.phoneme2id[p] = idx
                self.id2phoneme[idx] = p
        self.idx = len(self.phoneme2id)

    def encode(self, phonemes):
        return [self.phoneme2id.get(p, 1) for p in phonemes]

    def decode(self, token_ids):
        decoded = []
        prev = -1
        for tid in token_ids:
            if isinstance(tid, torch.Tensor): tid = tid.item()
            if tid not in (0, 1) and tid != prev:
                if tid in self.id2phoneme:
                    decoded.append(self.id2phoneme[tid])
            prev = tid
        return decoded

moshaf_rules = quran_transcript.MoshafAttributes(
    rewaya="hafs", madd_monfasel_len=4, madd_mottasel_len=4, 
    madd_mottasel_waqf=5, madd_aared_len=4
)

def normalize_initial_hamzat_wasl(text: str) -> str:
    t = text.strip()
    if t.startswith('ٱل') or t.startswith('ال'): return 'ءَ' + t[1:]
    elif t.startswith('ٱ') or t.startswith('ا'): return 'ءِ' + t[1:]
    return t

def safe_phonetizer(text, rules):
    norm = normalize_initial_hamzat_wasl(text)
    try: return list(quran_transcript.quran_phonetizer(norm, rules).phonemes)
    except Exception: return [c for c in norm if c not in '  \u064b\u064c\u064d\u064e\u064f\u0650\u0651\u0652\u0670']

print("✅ Phonetizer and Deterministic Vocabulary class initialized.")


✅ Phonetizer and Deterministic Vocabulary class initialized.


In [20]:
import os
import json
import io
import subprocess
import numpy as np
import soundfile as sf
from tqdm import tqdm

In [22]:
a = quran_transcript.Aya(1, 1)
print([x for x in dir(a) if not x.startswith("_")])
print(a.get())   # if .get() exists, this shows the available fields

['aya_idx', 'bismillah_imlaey_key', 'bismillah_map_key', 'bismillah_uthmani_key', 'decoding_cache', 'get', 'get_ayat_after', 'get_by_imlaey_words', 'get_formatted_rasm_map', 'get_start_imlaey_word_idx', 'imlaey_key', 'imlaey_to_uthmani', 'is_last', 'istiaatha_imlaey', 'istiaatha_uthmani', 'join_prefix', 'map_key', 'quran_dict', 'quran_path', 'save_quran_dict', 'set', 'set_new', 'set_rasm_map', 'start_imlaey_word_idx', 'step', 'step_by_imlaey_words', 'sura_idx', 'sura_name_key', 'uthmani_key']
AyaFormat(sura_idx=1, aya_idx=1, sura_name='الفاتحة', num_ayat_in_sura=7, uthmani='بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ', uthmani_words=['بِسْمِ', 'ٱللَّهِ', 'ٱلرَّحْمَـٰنِ', 'ٱلرَّحِيمِ'], imlaey_words=['بِسْمِ', 'اللَّهِ', 'الرَّحْمَٰنِ', 'الرَّحِيمِ'], imlaey='بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ', istiaatha_uthmani='أَعُوذُ بِٱللَّهِ مِنَ ٱلشَّيْطَانِ ٱلرَّجِيمِ', istiaatha_imlaey='أَعُوذُ بِاللَّهِ مِنَ الشَّيْطَانِ الرَّجِيمِ', rasm_map=None, bismillah_uthmani=None, bismillah_imlaey=Non

In [9]:
# ==============================================================================
# CELL 4: Data Fetching & Preprocessing (Full Ayahs)
# ==============================================================================
import kagglehub
kagglehub.login()
kaggle_path = kagglehub.dataset_download('ami0nai/quran-dataset')

def find_data_paths(base_dirs=[kaggle_path, "/kaggle/input", "."]):
    meta_path, audio_dir = None, None
    for base in base_dirs:
        for root, dirs, files in os.walk(base):
            for f in files:
                if f.endswith(".json") and "metadata" in f:
                    meta_path = os.path.join(root, f)
            for d in dirs:
                if d.lower() == "audio":
                    audio_dir = os.path.join(root, d)
            if meta_path and audio_dir: return meta_path, audio_dir
    return None, None

json_file, audio_folder = find_data_paths()
print(f"📄 Metadata JSON: {json_file}\n🔊 Audio Folder:  {audio_folder}")

with open(json_file, 'r', encoding='utf-8') as f:
    raw_metadata = json.load(f)

def load_audio_safely(path, target_sr=16000):
    try: y, sr = sf.read(path, dtype='float32')
    except Exception:
        cmd = ["ffmpeg", "-y", "-i", path, "-ar", str(target_sr), "-ac", "1", "-f", "wav", "-"]
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        out, _ = proc.communicate()
        import io
        y, sr = sf.read(io.BytesIO(out), dtype='float32')

    if len(y.shape) > 1: y = np.mean(y, axis=1)
    if sr != target_sr:
        import librosa
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
    max_val = np.max(np.abs(y))
    if max_val > 0: y = (y / max_val) * 0.95
    return y

valid_samples = []
all_phonemes_set = set()

for row in tqdm(raw_metadata, desc="Extracting Ayahs"):
    fname = row['audio']['path'] if isinstance(row['audio'], dict) else row['audio']
    apath = os.path.join(audio_folder, fname)
    if not os.path.exists(apath): continue
    
    uthmani = quran_transcript.Aya(row['surah_id'], row['ayah_id']).get().uthmani
    ph = safe_phonetizer(uthmani, moshaf_rules)
    if not ph: continue
    
    try:
        wav = load_audio_safely(apath, target_sr=16000)
        valid_samples.append({'audio': wav, 'phonemes': ph, 'text': uthmani})
        for p in ph: all_phonemes_set.add(p)
    except Exception: continue

vocab = DeterministicVocabulary(all_phonemes_set)
print(f"✅ Extracted {len(valid_samples)} full Ayahs.")


📄 Metadata JSON: /kaggle/input/datasets/ami0nai/quran-dataset/quran_full_data_collection_19_1008_murattal/quran_metadata.json
🔊 Audio Folder:  /kaggle/input/datasets/ami0nai/quran-dataset/quran_full_data_collection_19_1008_murattal/audio


Extracting Ayahs:   0%|          | 0/1232 [00:00<?, ?it/s]

[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (736) too large for available bit count (728)
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (928) too large for available bit count (920)
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (2016) too large for available bit count (1856)
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (768) too large for available bit count (736)
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (960) too large for available bit count (928)
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (992) too large for available bit count (928)
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (1984) too large for available bit count (1848)
[src/libmpg123/laye

✅ Extracted 1008 full Ayahs.


In [24]:
# ==============================================================================
# CELL 5: Streaming Chunk Generation (The "Beat" Mimicry)
# ==============================================================================
# Values mapped exactly to your production server logic
SAMPLE_RATE = 16000
CHUNK_WINDOW_SAMPLES = 48000  # 3.0 seconds (Trailing window for decode_live_window)
STRIDE_SAMPLES = 8000         # 0.5 seconds (MIN_NEW_BYTES beat size)

def generate_streaming_chunks(valid_samples, vocab):
    chunked_dataset = []
    
    for sample in tqdm(valid_samples, desc="Slicing audio into 3s server chunks"):
        audio = sample['audio']
        phonemes = sample['phonemes']
        
        # Skip if audio is shorter than the 3s window or has no targets
        if len(audio) < CHUNK_WINDOW_SAMPLES or len(phonemes) == 0:
            continue
            
        # Linear Time Alignment: Estimate how many audio samples represent 1 phoneme
        samples_per_phoneme = len(audio) / len(phonemes)
        
        # Slide the window exactly like tick() does every 0.5s
        for start_samp in range(0, len(audio) - CHUNK_WINDOW_SAMPLES + 1, STRIDE_SAMPLES):
            end_samp = start_samp + CHUNK_WINDOW_SAMPLES
            chunk_audio = audio[start_samp:end_samp]
            
            # Find which phonemes fall inside this 3-second time window
            start_ph_idx = max(0, int(start_samp / samples_per_phoneme))
            end_ph_idx = min(len(phonemes), int(end_samp / samples_per_phoneme) + 1)
            
            chunk_phonemes = phonemes[start_ph_idx:end_ph_idx]
            
            # Ensure the chunk has actual speech/targets to train on
            if len(chunk_phonemes) > 0:
                chunked_dataset.append({
                    'audio': chunk_audio,  # Exactly 48,000 samples
                    'phonemes': chunk_phonemes,
                    'targets': torch.tensor(vocab.encode(chunk_phonemes), dtype=torch.long)
                })
                
    return chunked_dataset

streaming_chunks = generate_streaming_chunks(valid_samples, vocab)
print(f"✅ Generated {len(streaming_chunks)} streaming chunks! (3s duration, 0.5s stride)")


Slicing audio into 3s server chunks: 100%|██████████| 1008/1008 [00:00<00:00, 11713.17it/s]

✅ Generated 8035 streaming chunks! (3s duration, 0.5s stride)


In [10]:
# ==============================================================================
# CELL 6: Model Architecture & Chunk DataLoader
# ==============================================================================
MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"
extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

class QuranWav2Vec2(nn.Module):
    def __init__(self, model_id, vocab_size):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_id)
        self.base.feature_extractor._freeze_parameters()
        self.base.config.gradient_checkpointing = False
        self.base.gradient_checkpointing_disable()

        self.phoneme_head = nn.Linear(self.base.config.hidden_size, vocab_size)
        self.ctc_loss = nn.CTCLoss(blank=1, zero_infinity=True)

    def forward(self, input_values, attention_mask=None, targets=None, target_lengths=None):
        outputs = self.base(input_values, attention_mask=attention_mask)
        logits = self.phoneme_head(outputs.last_hidden_state)

        if targets is not None:
            log_probs = logits.transpose(0, 1).log_softmax(2)
            input_lengths = torch.full((log_probs.size(1),), log_probs.size(0), dtype=torch.long, device=log_probs.device)
            loss = self.ctc_loss(log_probs, targets, input_lengths, target_lengths)
            return loss.unsqueeze(0), logits

        return logits

class StreamingChunkDataset(Dataset):
    def __init__(self, chunk_records): self.data = chunk_records
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

def chunk_collate_fn(batch):
    audios = [b['audio'] for b in batch]
    targets = [b['targets'] for b in batch]

    inputs = extractor(audios, sampling_rate=16000, return_tensors="pt")
    padded_targets = torch.nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=0)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)

    return inputs.input_values, inputs.attention_mask, padded_targets, target_lengths

print("✅ Model and Dataloaders configured for 3s chunks.")


preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

✅ Model and Dataloaders configured for 3s chunks.


In [26]:
# ==============================================================================
# CELL 7: Chunk-Based Training & Evaluation Loop
# ==============================================================================
from sklearn.model_selection import train_test_split
import Levenshtein

CHUNK_BATCH_SIZE = 16 if NUM_GPUS > 0 else 4
EPOCHS = 8

# Split data into Train and Validation
train_chunks, val_chunks = train_test_split(streaming_chunks, test_size=0.1, random_state=42)

train_loader = DataLoader(StreamingChunkDataset(train_chunks), batch_size=CHUNK_BATCH_SIZE, shuffle=True, collate_fn=chunk_collate_fn)
val_loader = DataLoader(StreamingChunkDataset(val_chunks), batch_size=CHUNK_BATCH_SIZE, shuffle=False, collate_fn=chunk_collate_fn)

print(f"📦 Training on {len(train_chunks)} chunks | Validating on {len(val_chunks)} chunks.")

model = QuranWav2Vec2(MODEL_ID, vocab.idx).to(DEVICE)
if NUM_GPUS > 1: model = nn.DataParallel(model)
core_model = model.module if isinstance(model, nn.DataParallel) else model

optimizer = torch.optim.AdamW([
    {'params': core_model.base.encoder.parameters(), 'lr': 3e-5},
    {'params': core_model.phoneme_head.parameters(), 'lr': 3e-4}
])

best_val_loss = float('inf')

for epoch in range(EPOCHS):
    # --- TRAINING ---
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    
    for input_vals, att_mask, targets, target_lens in pbar:
        optimizer.zero_grad()
        loss, _ = model(input_vals.to(DEVICE), None, targets.to(DEVICE), target_lens.to(DEVICE))
        loss = loss.mean()
        
        if torch.isnan(loss): continue
            
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    distances, total_ph = 0, 0
    
    with torch.no_grad():
        for input_vals, att_mask, targets, target_lens in tqdm(val_loader, desc=f"Epoch {epoch+1} [Eval]", leave=False):
            loss, logits = model(input_vals.to(DEVICE), None, targets.to(DEVICE), target_lens.to(DEVICE))
            val_loss += loss.mean().item()
            
            # Calculate Phoneme Error Rate (PER) on chunks
            pred_ids = torch.argmax(logits, dim=-1)
            for i in range(len(pred_ids)):
                p_text = "".join(vocab.decode(pred_ids[i]))
                t_text = "".join(vocab.decode(targets[i][:target_lens[i]]))
                distances += Levenshtein.distance(p_text, t_text)
                total_ph += len(t_text)

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    per = (distances / max(total_ph, 1)) * 100.0

    print(f"🏁 Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val PER: {per:.2f}%")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({'model_state_dict': core_model.state_dict()}, 'streaming_quran_wav2vec.pt')
        print(f"💾 Saved Best Model at Epoch {epoch+1}")


📦 Training on 7231 chunks | Validating on 804 chunks.


pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-arabic
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

...

🏁 Epoch 6 | Train Loss: 0.9251 | Val Loss: 0.8543 | Val PER: 32.42%
💾 Saved Best Model at Epoch 6


Epoch 7/8 [Train]: 100%|██████████| 452/452 [08:51<00:00,  1.18s/it, loss=0.7957]


🏁 Epoch 7 | Train Loss: 0.8634 | Val Loss: 0.8313 | Val PER: 33.56%
💾 Saved Best Model at Epoch 7


Epoch 8/8 [Train]: 100%|██████████| 452/452 [08:52<00:00,  1.18s/it, loss=0.9510]


🏁 Epoch 8 | Train Loss: 0.7997 | Val Loss: 0.7952 | Val PER: 31.74%
💾 Saved Best Model at Epoch 8


In [27]:
# ==============================================================================
# CELL 8: Simulate Server Socket & Live Inference
# ==============================================================================
def simulate_server_tick(audio_stream, model, vocab, extractor):
    """
    Simulates the logic in ml_engine.py and main.py:
    - Bytes arrive in a growing buffer.
    - Every 0.5 seconds (`STRIDE_SAMPLES`), if enough new bytes are present, it evaluates.
    - It extracts the trailing 3 seconds (`CHUNK_WINDOW_SAMPLES`) and runs inference.
    """
    model.eval()
    
    eval_pos = 0
    buffer = []
    
    print("\n🎧 Starting Simulated WebSocket Stream...")
    
    # We feed the audio into the buffer in tiny chunks (simulating network packets)
    packet_size = 4000  # 0.25s chunks
    for i in range(0, len(audio_stream), packet_size):
        buffer.extend(audio_stream[i : i + packet_size])
        
        # SERVER LOGIC: if len(audio) - eval_pos >= MIN_NEW_BYTES (8000 samples)
        if len(buffer) - eval_pos >= STRIDE_SAMPLES:
            # Update eval_pos (consume bytes)
            eval_pos = len(buffer)
            
            # Get trailing 3-second window
            trailing_start = max(0, len(buffer) - CHUNK_WINDOW_SAMPLES)
            live_window = np.array(buffer[trailing_start : len(buffer)], dtype=np.float32)
            
            # Simple RMS Gate (Speech vs Silence)
            rms = np.sqrt(np.mean(live_window**2))
            if rms < 0.01:  # Silence threshold
                print(f"⏱️ Tick @ {len(buffer)/SAMPLE_RATE:.1f}s | 🔇 Silence (RMS: {rms:.4f})")
                continue
                
            # Run Inference on the trailing window
            inputs = extractor([live_window], sampling_rate=16000, return_tensors="pt")
            with torch.no_grad():
                logits = model(inputs.input_values.to(DEVICE))
                pred_ids = torch.argmax(logits[0], dim=-1)
                
            # Decode the tokens (CTC collapse)
            phonemes = vocab.decode(pred_ids)
            live_text = " ".join(phonemes)
            
            print(f"⏱️ Tick @ {len(buffer)/SAMPLE_RATE:.1f}s | 🔊 Live: [{live_text}]")

# Test the simulation on a random full Ayah from the validation set
test_audio = valid_samples[0]['audio']
print(f"Target Full Ayah: {' '.join(valid_samples[0]['phonemes'])}")
simulate_server_tick(test_audio, core_model, vocab, extractor)


Target Full Ayah: ب ِ س م ِ   ل ل َ ا ا ه ِ   ر ر َ ح م َ ا ا ن ِ   ر ر َ ح ِ ۦ ۦ ۦ ۦ م

🎧 Starting Simulated WebSocket Stream...
⏱️ Tick @ 0.5s | 🔇 Silence (RMS: 0.0044)
⏱️ Tick @ 1.0s | 🔊 Live: [ب ِ]
⏱️ Tick @ 1.5s | 🔊 Live: [ب ِ س م]
⏱️ Tick @ 2.0s | 🔊 Live: [ب ِ س َ م ِ   ل ل َ]
⏱️ Tick @ 2.5s | 🔊 Live: [ب ِ س م ِ   ل ل َ ا ا ه ِ]
⏱️ Tick @ 3.0s | 🔊 Live: [ب ِ س م ِ   ل ل َ ا ا ه ِ   ر ر َ ح]
⏱️ Tick @ 3.5s | 🔊 Live: [س م ِ   ل ل َ ا ا ه ِ   ر ر َ ح م َ ا]
⏱️ Tick @ 4.0s | 🔊 Live: [ِ   ل ل َ ا ا ه ِ   ر ر َ ح م َ ا ن ِ]
⏱️ Tick @ 4.5s | 🔊 Live: [ل َ ا ا ه ِ   ر ر َ ح م َ ا ا ن ِ   ر ر]
⏱️ Tick @ 5.0s | 🔊 Live: [ِ   ر ر َ ح م َ ا ا ن ِ   ر ر َ ح ِ]
⏱️ Tick @ 5.5s | 🔊 Live: [ر َ ح م َ ا ا ن ِ   ر ر َ ح ِ]
⏱️ Tick @ 6.0s | 🔊 Live: [َ ا ن ِ   ر ر َ ح ِ ۦ ۦ م]


In [28]:
import os
print(os.getcwd())
print(os.path.getsize("streaming_quran_wav2vec.pt") / 1e6, "MB")

/kaggle/working
1262.095695 MB


In [37]:
   from IPython.display import FileLink
   FileLink("streaming_quran_wav2vec.pt")

/kaggle/working/streaming_quran_wav2vec.pt

In [30]:
import json
with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab.phoneme2id, f, ensure_ascii=False, indent=2)

In [35]:
import os
print(os.path.exists("/kaggle/working/my_recording.wav"))
!find /kaggle/input -iname "*.wav" -o -iname "*.mp3" -o -iname "*.m4a" -o -iname "*.ogg" | grep -v quran-dataset | head

False
/kaggle/input/datasets/ami0nai/my-recording/my_recording.wav


In [36]:
def phoneme_error_rate(model, wav, surah, ayah):
    target = safe_phonetizer(quran_transcript.Aya(surah, ayah).get().uthmani, moshaf_rules)
    model.eval()
    inputs = extractor([wav], sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = model(inputs.input_values.to(DEVICE))
    pred = vocab.decode(torch.argmax(logits[0], dim=-1))

    # map tokens to single characters so Levenshtein works on phoneme units
    enc = lambda toks: "".join(chr(0x100 + vocab.phoneme2id.get(t, 1)) for t in toks)
    per = Levenshtein.distance(enc(pred), enc(target)) / max(len(target), 1) * 100
    print("Target:", " ".join(target))
    print("Pred:  ", " ".join(pred))
    print(f"PER: {per:.1f}%")

path = "/kaggle/input/datasets/ami0nai/my-recording/my_recording.wav"
wav = load_audio_safely(path)
print(len(wav) / 16000, "seconds")

phoneme_error_rate(core_model, wav, surah=111, ayah=1)   # change to the ayah you recited
simulate_server_tick(wav, core_model, vocab, extractor)

7.68 seconds
Target: ت َ ب ب َ ت   ي َ د َ ا ا ا ا   ء َ ب ِ ۦ ۦ   ل َ ه َ ب ِ و و و َ ت َ ب ب ڇ
Pred:   ت ب ب َ ت   ي َ د َ ا ا   ء َ ب ِ ۦ ۦ   ل َ ه َ ب ِ و و َ ت َ ب ڇ
PER: 13.2%

🎧 Starting Simulated WebSocket Stream...
⏱️ Tick @ 0.5s | 🔇 Silence (RMS: 0.0055)
⏱️ Tick @ 1.0s | 🔊 Live: [ت َ ب ب]
⏱️ Tick @ 1.5s | 🔊 Live: [ت َ ب ب َ]
⏱️ Tick @ 2.0s | 🔊 Live: [ت َ ب ب َ ت ِ ي َ]
⏱️ Tick @ 2.5s | 🔊 Live: [ت َ ب ب َ ت   ي َ د َ ا ا]
⏱️ Tick @ 3.0s | 🔊 Live: [ت َ ب ب َ ت   ي َ د َ]
⏱️ Tick @ 3.5s | 🔊 Live: [ب ب َ ت   ي َ د َ]
⏱️ Tick @ 4.0s | 🔊 Live: [ب ب َ ت   ي َ د َ ا ا   ء َ]
⏱️ Tick @ 4.5s | 🔊 Live: [ت ي َ د َ ا ا   ء َ ب ِ]
⏱️ Tick @ 5.0s | 🔊 Live: [د َ ا   ء َ ب ِ ۦ ۦ   ل َ]
⏱️ Tick @ 5.5s | 🔊 Live: [ا   ء َ ب ِ ۦ ۦ   ل َ ه]
⏱️ Tick @ 6.0s | 🔊 Live: [ا ا   ء َ ب ِ ۦ ۦ   ل َ ه َ ب ِ و]
⏱️ Tick @ 6.5s | 🔊 Live: [ء َ ب ِ ۦ ۦ   ل َ ه َ ب ِ و و]
⏱️ Tick @ 7.0s | 🔊 Live: [ذ ِ ۦ ۦ   ل َ ه َ ب ِ و و َ ت َ ب]
⏱️ Tick @ 7.5s | 🔊 Live: [ۦ   ل َ ه َ ب ِ و و َ ت َ ب]


In [4]:
!find /kaggle/input -name "*.pt" -o -name "vocab.json"

/kaggle/input/models/ami0nai/quran-model/pytorch/default/1/streaming_quran_wav2vec (1).pt


In [11]:
CKPT = "/kaggle/input/models/ami0nai/quran-model/pytorch/default/1/streaming_quran_wav2vec (1).pt"

core_model = QuranWav2Vec2(MODEL_ID, vocab.idx).to(DEVICE)
ckpt = torch.load(CKPT, map_location=DEVICE)
core_model.load_state_dict(ckpt["model_state_dict"])
core_model.eval()
print("✅ Model loaded")

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-arabic
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded


In [14]:
import json
VOCAB = "/kaggle/input/models/ami0nai/quran-model/pytorch/default/1/vocab (2).json"  # confirm the real name
with open(VOCAB, encoding="utf-8") as f:
    saved = json.load(f)
print("Same vocab:", saved == vocab.phoneme2id)

Same vocab: True


In [17]:
import time
SAMPLE_RATE = 16000
CHUNK_WINDOW_SAMPLES = 48000  # 3.0 s
STRIDE_SAMPLES = 8000         # 0.5 s
core_model.eval()

# a fake 3-second window (or use a real one: valid_samples[0]['audio'][:48000])
window = np.random.randn(CHUNK_WINDOW_SAMPLES).astype(np.float32) * 0.1
inputs = extractor([window], sampling_rate=16000, return_tensors="pt")
x = inputs.input_values.to(DEVICE)

with torch.no_grad():
    for _ in range(3):                      # warm-up: the first runs are slower
        core_model(x)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    times = []
    for _ in range(10):
        t = time.time()
        core_model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()        # wait for the GPU to actually finish
        times.append(time.time() - t)

print(f"Device: {DEVICE}")
print(f"Average: {np.mean(times):.3f}s | Worst: {np.max(times):.3f}s")

Device: cuda
Average: 0.053s | Worst: 0.059s


In [18]:
cpu_model = QuranWav2Vec2(MODEL_ID, vocab.idx)
cpu_model.load_state_dict({k: v.cpu() for k, v in core_model.state_dict().items()})
cpu_model.eval()

x_cpu = x.cpu()
with torch.no_grad():
    cpu_model(x_cpu)  # warm-up
    t = time.time()
    for _ in range(3): cpu_model(x_cpu)
print(f"CPU average: {(time.time() - t) / 3:.3f}s")

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-arabic
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CPU average: 0.778s
